In [1]:
import geopandas as gpd
import pandas as pd

from keplergl import KeplerGl

# To install kepler 
do a `pip install keplergl`

It may complain that GDAL is not installed. If that happens then you need to install it. I followed the instructions here under option 2 `Install GDAL via brew` 

https://medium.com/@egiron/how-to-install-gdal-and-qgis-on-macos-catalina-ca690dca4f91


# Data sources:

https://raw.githubusercontent.com/ellcom/UK-Train-Station-Locations/master/uk-train-stations.csv <- stations data
https://datashare.ed.ac.uk/handle/10283/2423 <- rail network geometry file

Trainline Office location: 51.518080, -0.108460

In [2]:
stations = pd.read_csv("https://raw.githubusercontent.com/ellcom/UK-Train-Station-Locations/master/uk-train-stations.csv")
stations.head()

,3alpha,station_name,latitude,longitude
0,AAP,Alexandra Palace Rail Station,51.597925,-0.120210
1,AAT,Achanalt Rail Station,57.609576,-4.913846
2,ABA,Aberdare Rail Station,51.715060,-3.443095
3,ABC,Altnabreac Rail Station,58.388133,-3.706287
4,ABD,Aberdeen Rail Station,57.143687,-2.098693


In [3]:
network = gpd.read_file("/Users/alexwilkes/Downloads/UK_Railways/Railway.shp")
network = network.set_crs("EPSG:27700").to_crs("WGS84")
network = network[["geometry"]]
network.head()

DriverError: /Users/alexwilkes/Downloads/UK_Railways/Railway.shp: No such file or directory

In [ ]:
my_map = KeplerGl(
    height=800,
    data={"stations": stations},
)

my_map

In [ ]:
# Once you have the map set up how you want, you can access the config
my_config = my_map.config

# then pass this config back to KeplerGl to have the map always loads as you've set it
# e.g.
# my_map = KeplerGl(
#     height=800,
#     data={"geojson": geojson, "network": network, "stations": stations},
#     config=my_config
# )

# my_map

In [ ]:
# Extra - animations!
# Use this function to create a geojson that you can give to Kepler for animating linestrings in the data
def prepare_geojson(coords):
    template = """
     {
        "type": "FeatureCollection",
        "features": [
          {
            "type": "Feature",
            "properties": { "vendor":  "A",
            "vol":20},
            "geometry": {
              "type": "LineString",
              "coordinates":
                {{coords}}
            }
          }
        ]
      }
    """
    
    return template.replace("{{coords}}", str(coords))

coords = network.loc[11996, "geometry"].__geo_interface__["coordinates"]
coords = [[pair[0], pair[1], 0, 1564184363 + 10*i] for i, pair in enumerate(coords)]  

In [ ]:
geojson = prepare_geojson(coords)

# 🚂 Full UK & EU Rail Network 🗺️

In [ ]:
import os
import shutil
import urllib
from pathlib import Path
from typing import Any, Dict

import geopandas as gpd
import pandas as pd
from keplergl import KeplerGl

RAIL_MAP_CONFIG = {
    "version": "v1",
    "config": {
        "visState": {
            "filters": [],
            "layers": [
                {
                    "id": "7igl93y",
                    "type": "geojson",
                    "config": {
                        "dataId": "rail_network",
                        "label": "rail_network",
                        "color": [34, 63, 154],
                        "highlightColor": [252, 242, 26, 255],
                        "columns": {"geojson": "geometry"},
                        "isVisible": True,
                        "visConfig": {
                            "opacity": 0.8,
                            "strokeOpacity": 0.8,
                            "thickness": 0.4,
                            "strokeColor": [43, 143, 112],
                            "colorRange": {
                                "name": "Global Warming",
                                "type": "sequential",
                                "category": "Uber",
                                "colors": [
                                    "#5A1846",
                                    "#900C3F",
                                    "#C70039",
                                    "#E3611C",
                                    "#F1920E",
                                    "#FFC300",
                                ],
                            },
                            "strokeColorRange": {
                                "name": "Global Warming",
                                "type": "sequential",
                                "category": "Uber",
                                "colors": [
                                    "#5A1846",
                                    "#900C3F",
                                    "#C70039",
                                    "#E3611C",
                                    "#F1920E",
                                    "#FFC300",
                                ],
                            },
                            "radius": 10,
                            "sizeRange": [0, 10],
                            "radiusRange": [0, 50],
                            "heightRange": [0, 500],
                            "elevationScale": 5,
                            "enableElevationZoomFactor": True,
                            "stroked": True,
                            "filled": False,
                            "enable3d": False,
                            "wireframe": False,
                        },
                        "hidden": False,
                        "textLabel": [
                            {
                                "field": None,
                                "color": [255, 255, 255],
                                "size": 18,
                                "offset": [0, 0],
                                "anchor": "start",
                                "alignment": "center",
                            }
                        ],
                    },
                    "visualChannels": {
                        "colorField": None,
                        "colorScale": "quantile",
                        "strokeColorField": None,
                        "strokeColorScale": "quantile",
                        "sizeField": None,
                        "sizeScale": "linear",
                        "heightField": None,
                        "heightScale": "linear",
                        "radiusField": None,
                        "radiusScale": "linear",
                    },
                },
                {
                    "id": "iw84ku",
                    "type": "point",
                    "config": {
                        "dataId": "stations",
                        "label": "stations",
                        "color": [255, 250, 102],
                        "highlightColor": [252, 242, 26, 255],
                        "columns": {
                            "lat": "latitude",
                            "lng": "longitude",
                            "altitude": None,
                        },
                        "isVisible": True,
                        "visConfig": {
                            "radius": 3,
                            "fixedRadius": False,
                            "opacity": 0.8,
                            "outline": False,
                            "thickness": 2,
                            "strokeColor": None,
                            "colorRange": {
                                "name": "Global Warming",
                                "type": "sequential",
                                "category": "Uber",
                                "colors": [
                                    "#5A1846",
                                    "#900C3F",
                                    "#C70039",
                                    "#E3611C",
                                    "#F1920E",
                                    "#FFC300",
                                ],
                            },
                            "strokeColorRange": {
                                "name": "Global Warming",
                                "type": "sequential",
                                "category": "Uber",
                                "colors": [
                                    "#5A1846",
                                    "#900C3F",
                                    "#C70039",
                                    "#E3611C",
                                    "#F1920E",
                                    "#FFC300",
                                ],
                            },
                            "radiusRange": [0, 50],
                            "filled": True,
                        },
                        "hidden": False,
                        "textLabel": [],
                    },
                    "visualChannels": {
                        "colorField": None,
                        "colorScale": "quantile",
                        "strokeColorField": None,
                        "strokeColorScale": "quantile",
                        "sizeField": None,
                        "sizeScale": "linear",
                    },
                },
            ],
            "interactionConfig": {
                "tooltip": {
                    "fieldsToShow": {
                        "rail_network": [],
                        "stations": [{"name": "name", "format": None}],
                    },
                    "compareMode": False,
                    "compareType": "absolute",
                    "enabled": True,
                },
                "brush": {"size": 0.5, "enabled": False},
                "geocoder": {"enabled": False},
                "coordinate": {"enabled": False},
            },
            "layerBlending": "normal",
            "splitMaps": [],
            "animationConfig": {"currentTime": None, "speed": 1},
        },
        "mapState": {
            "bearing": 0,
            "dragRotate": False,
            "latitude": 48.377105772707104,
            "longitude": 23.44908342390212,
            "pitch": 0,
            "zoom": 4.084271774465849,
            "isSplit": False,
        },
        "mapStyle": {
            "styleType": "dark",
            "topLayerGroups": {},
            "visibleLayerGroups": {
                "label": True,
                "road": True,
                "border": False,
                "building": True,
                "water": True,
                "land": True,
                "3d building": False,
            },
            "threeDBuildingColor": [
                9.665468314072013,
                17.18305478057247,
                31.1442867897876,
            ],
            "mapStyles": {},
        },
    },
}


def get_rail_network_geodf(download: bool = False) -> gpd.GeoDataFrame:
    """Download or read pre-downloaded UK and EU rail network files as geo df.

    Uses data from https://www.diva-gis.org/gdata
    """

    COUNTRIES = [
        "GBR",
        "ALB",
        "AND",
        "AUT",
        "BEL",
        "BGR",
        "BIH",
        "CHE",
        "CYP",
        "CZE",
        "DEU",
        "DNK",
        "ESP",
        "EST",
        "FIN",
        "FRA",
        "GRC",
        "HRV",
        "HUN",
        "IRL",
        "ITA",
        "LTU",
        "LUX",
        "LVA",
        "MKD",
        "MLT",
        "MNE",
        "NLD",
        "NOR",
        "POL",
        "PRT",
        "ROU",  # manually download as file name does not match folder name ROU vs ROM
        "SRB",
        "SVK",
        "SVN",
        "SWE",
        "UKR",
    ]

    # Setup data directories
    ZIP_FILES_FOLDER = "rail_data/zip_files"
    UNZIP_FILES_FOLDER = "rail_data/unzip_files"

    Path(ZIP_FILES_FOLDER).mkdir(parents=True, exist_ok=True)
    Path(UNZIP_FILES_FOLDER).mkdir(parents=True, exist_ok=True)

    # Retrieve rail network data
    shapefiles = []
    for country in COUNTRIES:

        # Country file names
        country_zip = os.path.join(ZIP_FILES_FOLDER, f"{country}_rail_data.zip")
        country_unzip = os.path.join(UNZIP_FILES_FOLDER, f"{country}_rails.shp")

        if download:
            # Download zip rail data
            urllib.request.urlretrieve(
                f"https://biogeo.ucdavis.edu/data/diva/rrd/{country}_rrd.zip",
                country_zip,
            )

            # Unzip file
            shutil.unpack_archive(country_zip, UNZIP_FILES_FOLDER)

        # Save shapefile
        shapefiles.append(gpd.read_file(country_unzip))

    # Create rail network geodataframe
    gdf = gpd.GeoDataFrame(pd.concat(shapefiles))

    return gdf


def create_rail_map(
    download_data: bool = False, rail_map_config: Dict[str, Any] = RAIL_MAP_CONFIG
) -> KeplerGl:
    """Create an UK and EU rail map with stations and routes."""

    # Get rail network data
    gdf = get_rail_network_geodf(download=download_data)
    rail_network = gdf[["geometry"]]

    # Get stations data
    stations = pd.read_csv(
        "https://raw.githubusercontent.com/trainline-eu/stations/master/stations.csv",
        usecols=["id", "name", "latitude", "longitude"],
        sep=";",
    )

    # Create rail map
    rail_map = KeplerGl(
        height=800,
        data={"rail_network": rail_network, "stations": stations},
        config=rail_map_config,
    )

    return rail_map


rail_map = create_rail_map(download_data=True)
